# AI Virtual Assistant for Customer Service - Brev Launchable

## Content

* [Overview](#Overview)
* [Software Components](#Software-Components)
* [Key Functionality](#Key-Functionality)
* [How It Works](#How-It-Works)
* [Key Components](#Key-Components)
* [Prerequisites](#Prerequisites)
* [Deployment - hands on starts here](#Deployment)
* [Getting API Keys](#Getting-API-Keys)
* [Docker Compose Check](#Docker-Compose-Check)
* [Start the Docker Compose App](#Start-the-Docker-Compose-App)
* [Prepare Sample Data](#Prepare-Sample-Data)
* [Ingest Data](#Ingest-Data)
* [Open the User Interface](#Open-the-User-Interface)

## Overview

This notebook deploys the NVIDIA AI Virtual Assistant for Customer Service blueprint in an NVIDIA Brev Launchable. The lab version is optimized for a no-GPU VM: model inference uses NVIDIA-hosted NIMs, the app containers are pulled from public GHCR images, and Milvus runs with the CPU image.

The blueprint is a reference solution for a text-based virtual assistant. It combines retrieval-augmented generation, structured customer data, unstructured product documentation, conversation storage, sentiment analysis, and a sample UI. The assistant can answer general product questions and personalized customer-service questions by retrieving relevant records and documents before generating a response.

This Brev notebook walks through the Docker Compose deployment path. It is designed for lab participants: enter an NVIDIA API key, validate the Compose configuration, pull prebuilt images, start the services, prepare sample data, ingest that data, and open the UI.

## Software Components

- NVIDIA-hosted NIM microservices
  - Response generation: `nvidia/nemotron-3-nano-30b-a3b`
  - Embeddings: `nvidia/llama-nemotron-embed-1b-v2`
  - Reranking: `nvidia/llama-nemotron-rerank-1b-v2`
- Public GHCR app images for the agent, retrievers, analytics, API gateway, and UI
- Orchestrator agent built with LangGraph
- Text retrievers built with LangChain
- Structured data store: Postgres
- Unstructured vector store: CPU Milvus
- Sample customer-service UI exposed on port `3001`

## Key Functionality

- Personalized responses for structured and unstructured customer queries
- Multi-turn dialogue with conversation state
- RAG over product manuals, FAQs, customer profiles, and order history
- Sentiment analysis and conversation summarization
- Multi-session support with Redis and Postgres-backed persistence
- A sample web UI for testing the assistant

## How It Works

1. A user asks a customer-service question in the UI.
2. The agent decides whether it needs structured data, unstructured documents, or both.
3. Structured retrieval queries Postgres-backed customer/order data.
4. Unstructured retrieval embeds and searches documents stored in Milvus.
5. The agent uses retrieved context with hosted Nemotron 3 Nano to generate a grounded response.
6. Analytics services can summarize conversations and assess sentiment.

## Key Components

**Sample Data**

The repository includes synthetic customer profiles, order histories, FAQs, product manuals, and product catalog information for a customer-service scenario.

**AI Agent**

The agent uses LangGraph to coordinate sub-agents and tools. It calls hosted NVIDIA NIMs for response generation and reasoning.

**Structured Data Retriever**

The structured retriever works with Postgres and Vanna.AI-style query generation to answer customer and order-history questions.

**Unstructured Data Retriever**

The unstructured retriever chunks product manuals and FAQ PDFs, creates embeddings with a hosted NVIDIA embedding NIM, and stores vectors in CPU Milvus.

**Analytics and Admin Operations**

The analytics service provides reference APIs for summaries, sentiment, and stored conversation data.

![Blueprint Diagram](https://github.com/NVIDIA-AI-Blueprints/ai-virtual-assistant/raw/main/docs/imgs/IVA-blueprint-diagram-r5.png)

## Prerequisites

Before running the notebook, confirm the Brev instance page shows:

- The instance status is **Running**
- **VM Mode** has a green **Built** tag
- **script** has a green **Completed** tag
- The secure link for port `8889` is **Healthy**

You also need an NVIDIA API key for hosted NIM inference. No GPU, NGC Docker key, or GHCR login is required for the default lab path.

# Deployment

The hands-on portion starts here. Run each cell in order.

The startup script already cloned the repository, installed Jupyter, and prepared Docker. This notebook will write the local environment file and start Docker Compose only after you provide your NVIDIA API key.

## Getting API Keys

You need an NVIDIA API key for hosted NIM inference.

If you are attending a guided lab session, an NVIDIA API key may be provided to you by the instructor or lab environment. Use that key in the next cell when prompted for `NVIDIA_API_KEY`.

If an API key is not provided:

1. Go to [NVIDIA Build](https://build.nvidia.com/explore/discover).
2. Sign in.
3. Open the [Nemotron 3 Nano model page](https://build.nvidia.com/nvidia/nemotron-3-nano-30b-a3b).
4. Click **Get API Key** and generate a key.
5. Paste that key when the next cell prompts for `NVIDIA_API_KEY`.

For the default Brev lab path, you do not need to provide an NGC Docker key. The app images are public GHCR images.

In [ ]:
import getpass
import os

USE_GHCR_IMAGES = os.environ.get("USE_GHCR_IMAGES", "1").strip().lower() not in {"0", "false", "no"}
GHCR_OWNER = os.environ.get("GHCR_OWNER", "jspaulding-nv")
GHCR_IMAGE_PREFIX = os.environ.get("GHCR_IMAGE_PREFIX", "aiva-customer-service")
GHCR_TAG = os.environ.get("GHCR_TAG", "nemotron3-milvus-cpu")

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("Enter your NVIDIA API key: ")

if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY is required.")

NGC_API_KEY = os.environ.get("NGC_API_KEY", "")
if not USE_GHCR_IMAGES and not NGC_API_KEY:
    NGC_API_KEY = getpass.getpass("Docker/NGC API key (press Enter to reuse NVIDIA_API_KEY): ") or NVIDIA_API_KEY

if not USE_GHCR_IMAGES and not NGC_API_KEY:
    raise ValueError("NGC_API_KEY is required when building images locally from source.")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if NGC_API_KEY:
    os.environ["NGC_API_KEY"] = NGC_API_KEY
os.environ["USE_GHCR_IMAGES"] = "1" if USE_GHCR_IMAGES else "0"
os.environ["GHCR_OWNER"] = GHCR_OWNER
os.environ["GHCR_IMAGE_PREFIX"] = GHCR_IMAGE_PREFIX
os.environ["GHCR_TAG"] = GHCR_TAG

print("NVIDIA_API_KEY is set.")
if USE_GHCR_IMAGES:
    print(f"Using public GHCR app images: ghcr.io/{GHCR_OWNER}/{GHCR_IMAGE_PREFIX}-*:{GHCR_TAG}")
else:
    print("Using local source builds. NGC_API_KEY is set for nvcr.io pulls.")

## Locate Repository And Compose Files

This cell finds the cloned repository, selects the hosted-NIM Docker Compose file, and enables the GHCR override so Docker pulls prebuilt app images instead of building locally.

In [ ]:
from pathlib import Path
import os


def find_repo_root():
    start = Path.cwd().resolve()
    for path in (start, *start.parents):
        if (path / "deploy" / "compose" / "docker-compose.yaml").exists():
            return path

    for candidate in [Path.home() / "ai-virtual-assistant", Path("/home/ubuntu/ai-virtual-assistant")]:
        if (candidate / "deploy" / "compose" / "docker-compose.yaml").exists():
            return candidate

    raise FileNotFoundError("Could not find deploy/compose/docker-compose.yaml. Open this notebook from the ai-virtual-assistant repo.")


REPO_ROOT = find_repo_root()
COMPOSE_FILE = REPO_ROOT / "deploy" / "compose" / "docker-compose.yaml"
GHCR_COMPOSE_FILE = REPO_ROOT / "deploy" / "compose" / "docker-compose.ghcr.yaml"
ENV_FILE = REPO_ROOT / ".env.launchable"
os.chdir(REPO_ROOT)

COMPOSE_FILES = [COMPOSE_FILE]
if USE_GHCR_IMAGES:
    if not GHCR_COMPOSE_FILE.exists():
        raise FileNotFoundError(f"GHCR Compose override not found: {GHCR_COMPOSE_FILE}")
    COMPOSE_FILES.append(GHCR_COMPOSE_FILE)

COMPOSE_ARGS = []
for compose_file in COMPOSE_FILES:
    COMPOSE_ARGS.extend(["-f", str(compose_file)])

print(f"Repository root: {REPO_ROOT}")
print("Compose files:")
for compose_file in COMPOSE_FILES:
    print(f"- {compose_file}")

## Set Up The Environment File

The notebook writes `.env.launchable` in the repository root. This file stays local on the VM and is not baked into container images.

In [ ]:
env_lines = [
    f"NVIDIA_API_KEY={NVIDIA_API_KEY}",
    f"NGC_API_KEY={NGC_API_KEY}",
    "APP_LLM_MODELNAME=nvidia/nemotron-3-nano-30b-a3b",
    "APP_VECTORSTORE_INDEXTYPE=IVF_FLAT",
    f"USE_GHCR_IMAGES={'1' if USE_GHCR_IMAGES else '0'}",
    f"GHCR_OWNER={GHCR_OWNER}",
    f"GHCR_IMAGE_PREFIX={GHCR_IMAGE_PREFIX}",
    f"GHCR_TAG={GHCR_TAG}",
    "",
]
ENV_FILE.write_text("\n".join(env_lines), encoding="utf-8")
ENV_FILE.chmod(0o600)

print(f"Wrote environment file: {ENV_FILE}")

## Docker Compose Check

Confirm Docker Compose is available. If the default GHCR path is enabled, no registry login is required because the app images are public.

In [ ]:
import subprocess
import time
from collections import deque

LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
DEPLOY_LOG = LOG_DIR / "deploy_hosted_nims.log"


def docker_cmd(*args):
    probe = subprocess.run(["docker", "info"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if probe.returncode == 0:
        return ["docker", *args]
    return ["sudo", "docker", *args]


def run_logged(command, log_file, error_message):
    recent_lines = deque(maxlen=8)

    print(f"Running: {' '.join(str(part) for part in command)}", flush=True)
    print(f"Streaming Docker output to {log_file}", flush=True)
    last_progress = time.monotonic()

    with log_file.open("a", encoding="utf-8") as log:
        log.write(f"\n\n$ {' '.join(str(part) for part in command)}\n")
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            log.write(line)
            log.flush()
            stripped = line.strip()
            if stripped:
                recent_lines.append(stripped)
            now = time.monotonic()
            if now - last_progress >= 10:
                print(".", end="", flush=True)
                last_progress = now

        return_code = process.wait()

    print(" done", flush=True)

    if return_code != 0:
        print(error_message)
        print(f"Full Docker output: {log_file}")
        print("Last Docker output lines:")
        for line in recent_lines:
            print(line)
        raise RuntimeError(f"Command failed with exit code {return_code}.")

    return return_code


version = subprocess.run(docker_cmd("compose", "version"), text=True, capture_output=True, check=True)
print(version.stdout.strip())

if USE_GHCR_IMAGES:
    GHCR_USER = os.environ.get("GHCR_USER", "")
    GHCR_TOKEN = os.environ.get("GHCR_TOKEN", "")
    if GHCR_USER and GHCR_TOKEN:
        login = subprocess.run(
            docker_cmd("login", "ghcr.io", "-u", GHCR_USER, "--password-stdin"),
            input=GHCR_TOKEN,
            text=True,
            capture_output=True,
        )
        if login.returncode != 0:
            print(login.stdout)
            print(login.stderr)
            raise RuntimeError("Docker login to ghcr.io failed.")
        print("Docker is ready and authenticated with ghcr.io.")
    else:
        print("Docker is ready. GHCR images are public, so no GHCR login is required.")
else:
    login = subprocess.run(
        docker_cmd("login", "nvcr.io", "-u", "$oauthtoken", "--password-stdin"),
        input=NGC_API_KEY,
        text=True,
        capture_output=True,
    )
    if login.returncode != 0:
        print(login.stdout)
        print(login.stderr)
        raise RuntimeError("Docker login to nvcr.io failed. If you used a Build API key, use an NGC personal key for NGC_API_KEY.")

    print("Docker is ready and authenticated with nvcr.io.")

## Validate The Compose Configuration

This cell checks that Docker Compose is configured for hosted Nemotron 3 Nano, CPU Milvus, and public GHCR app images.

In [ ]:
config = subprocess.run(
    docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "config"),
    text=True,
    capture_output=True,
)
if config.returncode != 0:
    print(config.stdout)
    print(config.stderr)
    raise RuntimeError("Docker Compose config validation failed.")

rendered = config.stdout
assert "nvidia/nemotron-3-nano-30b-a3b" in rendered
assert "milvusdb/milvus:v2.4.15" in rendered
assert "v2.4.15-gpu" not in rendered
assert "KNOWHERE_GPU_MEM_POOL_SIZE" not in rendered

if USE_GHCR_IMAGES:
    assert f"ghcr.io/{GHCR_OWNER}/{GHCR_IMAGE_PREFIX}-agent:{GHCR_TAG}" in rendered
    assert "\nbuild:" not in rendered
    print("Docker Compose configuration validated for hosted Nemotron 3 Nano, CPU Milvus, and GHCR app images.")
else:
    print("Docker Compose configuration validated for hosted Nemotron 3 Nano and CPU Milvus.")

## Start The Docker Compose App

This cell pulls images and starts the services with `--no-build`. On a fresh VM, pulling and extracting images can still take several minutes.

Full Docker output is saved to `logs/deploy_hosted_nims.log`. To watch it live, open a Jupyter terminal and run:

```bash
tail -f ~/ai-virtual-assistant/logs/deploy_hosted_nims.log
```

In [ ]:
DEPLOY_LOG.write_text("", encoding="utf-8")

if USE_GHCR_IMAGES:
    print("Pulling public GHCR app images and upstream service images. This can take a few minutes on a fresh VM.", flush=True)
    run_logged(
        docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "pull"),
        DEPLOY_LOG,
        "Docker Compose image pull failed.",
    )
    print("Starting Docker Compose services without local builds.", flush=True)
    run_logged(
        docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "up", "-d", "--no-build"),
        DEPLOY_LOG,
        "Docker Compose deployment failed.",
    )
else:
    print("Building and starting Docker Compose services from local source. This can take several minutes.", flush=True)
    run_logged(
        docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "up", "-d", "--build"),
        DEPLOY_LOG,
        "Docker Compose deployment failed.",
    )

print("Deployment started. Full Docker output was saved to logs/deploy_hosted_nims.log.")

## Check Container Status

This cell prints the Docker Compose service table. Some services may still be starting immediately after `up -d`; refresh or rerun this cell if needed.

In [ ]:
ps = subprocess.run(
    docker_cmd("compose", "--env-file", str(ENV_FILE), *COMPOSE_ARGS, "ps"),
    text=True,
    capture_output=True,
)
print(ps.stdout)
if ps.returncode != 0:
    print(ps.stderr)

print("\nOpen the sample UI on port 3001 once services are healthy.")

## Prepare Sample Data

Download the sample product manuals so the ingestion notebook can load the PDF documents into Milvus.

In [ ]:
from urllib.parse import unquote, urlparse
from urllib.request import urlretrieve

manual_list = REPO_ROOT / "data" / "list_manuals.txt"
manual_dir = REPO_ROOT / "data" / "manuals_pdf"
manual_dir.mkdir(parents=True, exist_ok=True)

downloaded = []
skipped = []

for url in manual_list.read_text(encoding="utf-8").splitlines():
    url = url.strip()
    if not url or url.startswith("#"):
        continue

    filename = Path(unquote(urlparse(url).path)).name
    target = manual_dir / filename
    if target.exists() and target.stat().st_size > 0:
        skipped.append(filename)
        continue

    print(f"Downloading {filename}...")
    urlretrieve(url, target)
    downloaded.append(filename)

print(f"Manuals ready in {manual_dir}")
print(f"Downloaded {len(downloaded)} file(s); skipped {len(skipped)} existing file(s).")
print("Next: open notebooks/ingest_data.ipynb to load the sample structured and unstructured data.")

## Ingest Data

Open and run:

```text
notebooks/ingest_data.ipynb
```

That notebook loads:

- Product manuals and FAQs into Milvus
- Structured customer/order data into Postgres

After ingestion finishes, the UI can answer questions grounded in the sample data.

## Open The User Interface

Return to the Brev instance page and scroll to **Using Secure Links**. Open the URL for port `3001` when its health shows **Healthy**.

If the `3001` link is not healthy yet, wait another minute and refresh the Brev page. The app containers may still be starting.

Do not use `--profile local-nim` for this lab. The local NIM profile is for self-hosted models and requires GPUs.